In [1]:
import datetime as dt
from uuid import uuid4

import polars as pl

# 🥉 Bronze
## Parameters

In [2]:
file_directory = "../data/bronze/depdev/"
file = "PSGC-3Q-2025-Publication-Datafile.xlsx"
valid_from = "2025-10-13T00:00:00.000+0800"

table_name = "dim_barangay"

# 🥉Bronze -> 🥈Silver

In [3]:
# reading the excel file from bronze
df = pl.read_excel(source=file_directory + file, sheet_name="PSGC")
df.sample(10)

Could not determine dtype for column 4, falling back to string
Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 10, falling back to string


10-digit PSGC,Name,Correspondence Code,Geographic Level,Old names,City Class,Income Classification (DOF DO No. 074.2024),Urban / Rural (based on 2020 CPH),2024 Population,__UNNAMED__9,Status
str,str,i64,str,str,str,str,str,i64,str,str
"""0205009010""","""Didipio""",25009010,"""Bgy""",null,null,null,"""R""",4413,null,null
"""0804802009""","""San Pedro""",84802009,"""Bgy""",null,null,null,"""R""",1137,null,null
"""0403416023""","""Munting Kawayan""",43416023,"""Bgy""",null,null,null,"""R""",771,null,null
"""1900705007""","""Cabcaban""",150705007,"""Bgy""",null,null,null,"""R""",1353,null,null
"""1903621033""","""Talao""",153621033,"""Bgy""",null,null,null,"""R""",1538,null,null
"""1600301033""","""Gamao""",160301033,"""Bgy""",null,null,null,"""R""",1154,null,null
"""1999905003""","""Datu Binasing""",124711012,"""Bgy""",null,null,null,"""R""",1699,null,null
"""0403430007""","""San Benito""",43430007,"""Bgy""",null,null,null,"""R""",3614,null,null
"""1900706030""","""Silangkum""",150706030,"""Bgy""",null,null,null,"""R""",1397,null,null


In [4]:
# check columns
df.columns

['10-digit PSGC',
 'Name',
 'Correspondence Code',
 'Geographic Level',
 'Old names',
 'City Class',
 'Income\r\nClassification (DOF DO No. 074.2024)',
 'Urban / Rural\r\n(based on 2020 CPH)',
 '2024 Population',
 '__UNNAMED__9',
 'Status']

In [5]:
# build corrected column names
correct_columns = {
    "10-digit PSGC": "psgc_id",
    "Name": "psgc_name",
    "Correspondence Code": "correspondence_code",
    "Geographic Level": "geographic_level",
    "Old names": "old_name",
    "City Class": "city_class",
    "Income\r\nClassification (DOF DO No. 074.2024)": "income_classification",
    "Urban / Rural\r\n(based on 2020 CPH)": "settlement_type",
    "2024 Population": "population",
    "__UNNAMED__9": "remarks",
    "Status": "status",
}

In [6]:
# rename columns
renamed_df = df.rename(correct_columns)
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,i64,str,str,str,str,str,i64,str,str
"""1408101015""","""Eva Puzon""",148101015,"""Bgy""",null,null,null,"""U""",1059,null,null
"""0907201046""","""Kauswagan""",97201046,"""Bgy""","""Talisay""",null,null,"""R""",1872,null,null
"""1804611028""","""Poblacion""",74611028,"""Bgy""",null,null,null,"""U""",16901,null,null
"""0405649030""","""Punta""",45649030,"""Bgy""",null,null,null,"""R""",591,null,null
"""0802617041""","""Tawagan """,82617041,"""Bgy""",null,null,null,"""R""",1537,null,"""Pob."""
"""0105547021""","""Unzad""",15547021,"""Bgy""",null,null,null,"""R""",4005,null,null
"""0906604004""","""Bual""",156604004,"""Bgy""",null,null,null,"""R""",2638,null,null
"""0806013013""","""Laygayon""",86013013,"""Bgy""",null,null,null,"""R""",888,null,null
"""1908704000""","""Datu Blah T. Sinsuat""",153830000,"""Mun""",null,null,"""3rd""","""""",32088,null,null


## col: `psgc_id`

In [7]:
# verify that all psgc_id have length 10
assert len(renamed_df["psgc_id"].str.len_chars().value_counts()["psgc_id"])==1
assert renamed_df["psgc_id"].str.len_chars().value_counts().row(0)[0] == 10

renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,i64,str,str,str,str,str,i64,str,str
"""0403417023""","""Lawaguin""",43417023,"""Bgy""",null,null,null,"""R""",507,null,null
"""0600613003""","""Barangay 1 """,60613003,"""Bgy""",null,null,null,"""U""",2170,null,"""Pob."""
"""0201504027""","""Gabut""",21504027,"""Bgy""",null,null,null,"""R""",1103,null,null
"""0102902009""","""Guardia""",12902009,"""Bgy""",null,null,null,"""R""",524,null,null
"""0600608024""","""Guintas""",60608024,"""Bgy""",null,null,null,"""R""",2358,null,null
"""0402115028""","""San Roque""",42115028,"""Bgy""",null,null,null,"""R""",3099,null,null
"""0804819029""","""Barangay East """,84819029,"""Bgy""",null,null,null,"""R""",840,null,"""Pob."""
"""0701225024""","""Mayana""",71225024,"""Bgy""",null,null,null,"""R""",2277,null,null
"""0105536010""","""Casaratan""",15536010,"""Bgy""",null,null,null,"""R""",1277,null,null


## col: `correspondence_code`

In [8]:
# cast to string because right now they're i64
renamed_df = renamed_df.with_columns(
    pl.col("correspondence_code").cast(pl.Utf8)
)

In [9]:
# make sure all are 9 chars or 0 if it's empty
renamed_df = renamed_df.with_columns(pl.col("correspondence_code").fill_null(""))
renamed_df = renamed_df.with_columns(
    pl.when(pl.col("correspondence_code").str.len_chars() == 8)
    .then(pl.col("correspondence_code").str.zfill(9))
    .otherwise(pl.col("correspondence_code")),
)

In [10]:
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,str,i64,str,str
"""1606817006""","""Maglambing""","""166817006""","""Bgy""",null,null,null,"""R""",1723,null,null
"""0401023024""","""Muzon""","""041023024""","""Bgy""",null,null,null,"""R""",1753,null,null
"""0806009012""","""Garcia""","""086009012""","""Bgy""",null,null,null,"""R""",397,null,null
"""0804808022""","""Canyomanao""","""084808022""","""Bgy""",null,null,null,"""R""",497,null,null
"""0501734039""","""Serranzana""","""051734039""","""Bgy""",null,null,null,"""R""",1140,null,null
"""0907205000""","""Labason""","""097205000""","""Mun""",null,null,"""2nd""","""""",44615,null,null
"""0102914026""","""San Antonio""","""012914026""","""Bgy""",null,null,null,"""R""",1955,null,null
"""0701230056""","""Talisay""","""071230056""","""Bgy""",null,null,null,"""R""",1053,null,null
"""1402705011""","""Payawan""","""142705011""","""Bgy""",null,null,null,"""R""",1755,null,null


## col: `settlement_type`

In [11]:
# map to the valid enum
SettlementTypeEnum = pl.Enum(categories=["urban", "rural", "-"])
settlement_type_map = {"R": "rural", "U": "urban", "": None}
renamed_df = renamed_df.with_columns(
    pl.col("settlement_type").replace(settlement_type_map)
)
renamed_df = renamed_df.with_columns(pl.col("settlement_type").cast(SettlementTypeEnum))
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,str
"""1401109006""","""Sagpat""","""141109006""","""Bgy""",null,null,null,"""rural""",3451,null,null
"""0301421037""","""Poblacion""","""031421037""","""Bgy""",null,null,null,"""rural""",3183,null,null
"""0102817003""","""Nalvo""","""012817003""","""Bgy""","""Cababaan-Nalvo""",null,null,"""rural""",488,null,null
"""0401014057""","""San Carlos""","""041014057""","""Bgy""",null,null,null,"""urban""",7416,null,null
"""0405649032""","""Rizal Ilaya""","""045649032""","""Bgy""",null,null,null,"""rural""",340,null,null
"""0501711050""","""Terogo""","""051711050""","""Bgy""",null,null,null,"""rural""",1110,null,null
"""0405620002""","""Agos-agos""","""045620002""","""Bgy""",null,null,null,"""rural""",2780,null,null
"""0102914008""","""Cagayungan""","""012914008""","""Bgy""",null,null,null,"""rural""",1287,null,null
"""1381300103""","""San Vicente""","""137404103""","""Bgy""",null,null,null,"""urban""",7362,null,null


In [12]:
renamed_df["status"].value_counts()

status,count
str,u32
"""Pob.""",2773
"""Capital""",82
null,40914


## col: `status`

In [13]:
BarangayStatusEnum = pl.Enum(categories=["poblacion", "capital"])
barangay_status_map = {
    "Pob.": "poblacion",
    "Capital": "capital",
}
renamed_df = renamed_df.with_columns(pl.col("status").replace(barangay_status_map))
renamed_df = renamed_df.with_columns(pl.col("status").cast(BarangayStatusEnum))
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,enum
"""0603047001""","""Balud Lilo-an""","""063047001""","""Bgy""",null,null,null,"""rural""",768,null,null
"""0331400000""","""City of Olongapo""","""037107000""","""City""",null,"""HUC""","""1st""",null,264903,null,null
"""0600609005""","""Cadajug""","""060609005""","""Bgy""",null,null,null,"""rural""",1295,null,null
"""0405610042""","""Tagbacan Ibaba""","""045610042""","""Bgy""",null,null,null,"""rural""",2285,null,null
"""0102812079""","""Bgy. No. 29, Santo Tomas ""","""012812079""","""Bgy""",null,null,null,"""rural""",1511,null,"""poblacion"""
"""0701240009""","""Cahayag""","""071240009""","""Bgy""",null,null,null,"""rural""",516,null,null
"""0401032007""","""Mabayabas""","""041032007""","""Bgy""",null,null,null,"""rural""",2331,null,null
"""0504109007""","""Labangtaytay""","""054109007""","""Bgy""",null,null,null,"""rural""",909,null,null
"""1704001059""","""Tumagabok""","""174001059""","""Bgy""",null,null,null,"""rural""",467,null,null


## col: `old_name`, `remarks`

In [14]:
renamed_df = renamed_df.with_columns(
    [
        pl.col("old_name").fill_null(""),
        pl.col("remarks").fill_null(""),
    ]
)
renamed_df.sample(5)

psgc_id,psgc_name,correspondence_code,geographic_level,old_name,city_class,income_classification,settlement_type,population,remarks,status
str,str,str,str,str,str,str,enum,i64,str,enum
"""1380610025""","""Barangay 710""","""133910025""","""Bgy""","""""",null,null,"""urban""",747,"""""",null
"""1004208012""","""Luzaran""","""104208012""","""Bgy""","""""",null,null,"""rural""",584,"""""",null
"""0600407008""","""Estancia""","""060407008""","""Bgy""","""""",null,null,"""urban""",9888,"""""",null
"""1381200008""","""Kapasigan""","""137403008""","""Bgy""","""""",null,null,"""urban""",4952,"""""",null
"""0500513002""","""Alabangpuro""","""050513002""","""Bgy""","""""",null,null,"""rural""",888,"""""",null


## filter: `brgy` only

In [15]:
renamed_df = renamed_df.filter(pl.col("geographic_level")=="Bgy")

In [16]:
renamed_df = renamed_df.drop(
    [
        "city_class",
        "income_classification",
        "population",
        "geographic_level",
    ],
    strict=False
)

# 🥈 Silver

In [17]:
renamed_df.sample(10)

psgc_id,psgc_name,correspondence_code,old_name,settlement_type,remarks,status
str,str,str,str,enum,str,enum
"""0802605033""","""Rawis""","""082605033""","""""","""rural""","""""",null
"""0501718042""","""Mambulo Nuevo""","""051718042""","""""","""urban""","""""",null
"""0701214010""","""Candajec""","""071214010""","""""","""rural""","""""",null
"""0203107008""","""Del Corpuz""","""023107008""","""""","""rural""","""""",null
"""0603031016""","""Singay""","""063031016""","""""","""rural""","""""",null
"""0701203011""","""Santa Cruz""","""071203011""","""""","""rural""","""""",null
"""0603001018""","""Pili""","""063001018""","""""","""rural""","""""",null
"""0701232002""","""Agahay""","""071232002""","""""","""rural""","""""",null
"""0504113014""","""Mapuyo""","""054113014""","""""","""rural""","""""",null


In [18]:
silver = renamed_df

# 🥈Silver -> 🥇 Gold

In [19]:
import blake3

silver = silver.with_columns(
    pl.struct(["psgc_name", "psgc_id"])
    .map_elements(
        lambda row: blake3.blake3(
            f"{row["psgc_name"]}_{row["psgc_id"]}".encode(encoding="utf-8")
        ).hexdigest()
    )
    .alias("identity_hash"),
    pl.struct(
        ["settlement_type", "status", "remarks", "correspondence_code", "old_name"]
    )
    .map_elements(
        lambda row: blake3.blake3(
            (
                f"{row["settlement_type"]}_{row["status"]}_{row["remarks"]}"
                + f"_{row["correspondence_code"]}_{row["old_name"]}"
            ).encode(encoding="utf-8")
        ).hexdigest()
    )
    .alias("fields_hash"),
)

In [20]:
release_date = pl.Series([valid_from]).str.strptime(
    pl.Datetime, "%Y-%m-%dT%H:%M:%S%.3f%z"
)[0]
silver = silver.with_columns(
    pl.lit(release_date).alias("valid_from"),
)
silver = silver.with_columns(
    pl.lit(dt.datetime.now(tz=dt.timezone.utc)).alias("ingestion_datetime"),
    pl.Series(name="surrogate_id", values=[str(uuid4()) for _ in range(len(silver))])
    .cast(pl.String)
    .alias("surrogate_id"),
)

# ordering columns
silver = silver.select(
    [
        "surrogate_id",
        "ingestion_datetime",
        "psgc_id",
        "psgc_name",
        "correspondence_code",
        "old_name",
        "settlement_type",
        "status",
        "remarks",
        "identity_hash",
        "fields_hash",
        "valid_from",
    ]
)

In [21]:
gold = silver

# 🥇 Gold

In [22]:
gold.sample(5)

surrogate_id,ingestion_datetime,psgc_id,psgc_name,correspondence_code,old_name,settlement_type,status,remarks,identity_hash,fields_hash,valid_from
str,"datetime[μs, UTC]",str,str,str,str,enum,enum,str,str,str,"datetime[μs, UTC]"
"""1b22d7d5-8d51-4608-8508-fe0f2f…",2025-10-18 08:58:37.195127 UTC,"""1001312031""","""Managok""","""101312031""","""""","""urban""",null,"""""","""3ac12017b118209932add1dbf1cdd3…","""b41f5d61c2a962f45c60020924e2a2…",2025-10-12 16:00:00 UTC
"""ed105d2a-a626-474e-8095-f53cb1…",2025-10-18 08:58:37.195127 UTC,"""0603012043""","""Pamuringao Garrido""","""063012043""","""""","""rural""",null,"""""","""75f55931336156e85a2f7703451e28…","""ca3a78e25aaeb29a9331986b0b5664…",2025-10-12 16:00:00 UTC
"""5d808d32-f9e5-4607-a2e3-31d1aa…",2025-10-18 08:58:37.195127 UTC,"""0802604003""","""Ando""","""082604003""","""""","""rural""",null,"""""","""aaf3e744cdd494df8c44b3fe680f48…","""fa2917523ffb3dd6f1828414b75010…",2025-10-12 16:00:00 UTC
"""20b434c6-a8e3-4524-af32-abd29d…",2025-10-18 08:58:37.195127 UTC,"""0806404015""","""Plaridel""","""086404015""","""""","""rural""",null,"""""","""8881cbd1c00b507cd8a5e0da746479…","""ae72e899469dae48d1984d3733dc7e…",2025-10-12 16:00:00 UTC
"""23acab45-8bef-41f0-8cf4-186b4a…",2025-10-18 08:58:37.195127 UTC,"""1206314008""","""Buto""","""126314008""","""""","""rural""",null,"""""","""1c48d7556ae6b36d59e9d06a6f2a8c…","""fb483d488d8286af88d73778369c63…",2025-10-12 16:00:00 UTC


In [23]:
from pap_datalab.engine import PapDatalab

lab = PapDatalab(environment="dev", environment_path="../dev.env")
client = lab.get_client(database="depdev")

In [24]:
for i in range(0, len(gold), 5000):
    chunk = gold[i:i+5000]
    client.insert_arrow(table=table_name, arrow_table=chunk.to_arrow())